# libs

In [1]:
!pip install unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 5.0 MB/s eta 0:00:0000:01


In [2]:
from google.colab import drive
import numpy as np
import pandas as pd
import re

from datetime import datetime, time, date
from unidecode import unidecode


# import dos dados

In [3]:
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [4]:
df_path = "/content/drive/MyDrive/projeto-integrador-I/data/processed/df_cleaned.csv"
df = pd.read_csv(df_path)
df.head()

,NOME_MUNICIPIO,DATA_REGISTRO,DATA_OCORRENCIA_BO,HORA_OCORRENCIA_BO,DESC_PERIODO,DESC_SUBTIPOLOCAL,BAIRRO,RUBRICA,MES_ESTATISTICA,ANO_ESTATISTICA,NOME_DELEGACIA_CIRCUNSCRICAO,NOME_DEPARTAMENTO_CIRCUNSCRICAO,NOME_SECCIONAL_CIRCUNSCRICAO,NOME_MUNICIPIO_CIRCUNSCRICAO
0,SANTOS,2022-01-05,2021-11-20 00:00:00,21:00:00,NaN,Comércio e Serviços,VILA MATIAS,Art. 213 - Estupro,1,2022,NaN,NaN,NaN,NaN
1,SANTOS,2022-01-09,2022-01-09 00:00:00,00:07:39,NaN,Residência,MARAPE,Estupro de vulneravel (art.217-A),1,2022,NaN,NaN,NaN,NaN
2,SANTOS,2022-01-16,2022-01-15 00:00:00,19:30:27,NaN,Residência,MACUCO,Art. 213 - Estupro,1,2022,NaN,NaN,NaN,NaN
3,SANTOS,2022-01-11,NaN,NaN,EM HORA INCERTA,Residência,MORRO NOVA CINTRA,Estupro de vulneravel (art.217-A),1,2022,NaN,NaN,NaN,NaN
4,SANTOS,2022-02-05,2021-10-17 00:00:00,NaN,DE MADRUGADA,Via Pública,MACUCO,Art. 213 - Estupro,2,2022,NaN,NaN,NaN,NaN


In [5]:
# uppercase em todo o dataframe

for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].str.upper()

df.head()

,NOME_MUNICIPIO,DATA_REGISTRO,DATA_OCORRENCIA_BO,HORA_OCORRENCIA_BO,DESC_PERIODO,DESC_SUBTIPOLOCAL,BAIRRO,RUBRICA,MES_ESTATISTICA,ANO_ESTATISTICA,NOME_DELEGACIA_CIRCUNSCRICAO,NOME_DEPARTAMENTO_CIRCUNSCRICAO,NOME_SECCIONAL_CIRCUNSCRICAO,NOME_MUNICIPIO_CIRCUNSCRICAO
0,SANTOS,2022-01-05,2021-11-20 00:00:00,21:00:00,NaN,COMÉRCIO E SERVIÇOS,VILA MATIAS,ART. 213 - ESTUPRO,1,2022,NaN,NaN,NaN,NaN
1,SANTOS,2022-01-09,2022-01-09 00:00:00,00:07:39,NaN,RESIDÊNCIA,MARAPE,ESTUPRO DE VULNERAVEL (ART.217-A),1,2022,NaN,NaN,NaN,NaN
2,SANTOS,2022-01-16,2022-01-15 00:00:00,19:30:27,NaN,RESIDÊNCIA,MACUCO,ART. 213 - ESTUPRO,1,2022,NaN,NaN,NaN,NaN
3,SANTOS,2022-01-11,NaN,NaN,EM HORA INCERTA,RESIDÊNCIA,MORRO NOVA CINTRA,ESTUPRO DE VULNERAVEL (ART.217-A),1,2022,NaN,NaN,NaN,NaN
4,SANTOS,2022-02-05,2021-10-17 00:00:00,NaN,DE MADRUGADA,VIA PÚBLICA,MACUCO,ART. 213 - ESTUPRO,2,2022,NaN,NaN,NaN,NaN


# periodos

In [6]:
# função de definição do periodo do dia

def set_period(value: str) -> str:
    """
    PT-BR: recebe um horário em string, converte para
           time.time() e retorna um perído com base no horário.

    params
    ----------
    value : str

        PT-BR: horário em string.

    return:
    ----------
    str
        horário convertido em período.
    """

    # verifica se é do tipo time.time()
    if isinstance(value, time):
        time_occ = value
    
    # se nao for, converte
    else:
        try:
            time_occ = datetime.strptime(str(value), '%H:%M:%S').time()
        except ValueError:
            return 'Horário inválido'
        

    # categorizando os periodos do dia
    if time(0, 0, 0) <= time_occ <= time(5, 59, 59):        # MADRUGADA, entre 0h e 5h 59min 59s
        return 'DE MADRUGADA'                               
    
    elif time(6, 0, 0) <= time_occ <= time(11, 59, 59):     # DE MANHÃ, entre 6h e 11h 59min 59s
        return 'DE MANHÃ'                                   
    
    elif time(12, 0, 0) <= time_occ <= time(17, 59, 59):    # À TARDE, entre 12h e 17h 59min 59s
        return 'À TARDE'                                    
    
    elif time(18, 0, 0) <= time_occ <= time(23, 59, 59):    # À NOITE, entre 18h e 23h 59min 59s
        return 'À NOITE'                                    
    

    # retorna horario invalido caso algum horario nao tenha sido inserido corretamente 
    else:
        return 'Horário inválido'

In [7]:
test = set_period("19:30:27")
test

'À NOITE'

In [8]:
# aplicando a conversão de hora para período na coluna 'DESC_PERIODO'

df['DESC_PERIODO'] = df.apply(
    lambda row: set_period(row['HORA_OCORRENCIA_BO'])

    # se o valor da coluna de periodos for nulo, aplica a função de definição de períodos nela
    if pd.isna(row['DESC_PERIODO']) or row['DESC_PERIODO'] == ''

    # senão, deixa o valor como está, mantendo o período registrado
    else row['DESC_PERIODO'],
    axis=1
)

# apaga a os horários porque não serão mais necessárias
df_standard_time = df.drop(columns='HORA_OCORRENCIA_BO')
df_standard_time

,NOME_MUNICIPIO,DATA_REGISTRO,DATA_OCORRENCIA_BO,DESC_PERIODO,DESC_SUBTIPOLOCAL,BAIRRO,RUBRICA,MES_ESTATISTICA,ANO_ESTATISTICA,NOME_DELEGACIA_CIRCUNSCRICAO,NOME_DEPARTAMENTO_CIRCUNSCRICAO,NOME_SECCIONAL_CIRCUNSCRICAO,NOME_MUNICIPIO_CIRCUNSCRICAO
0,SANTOS,2022-01-05,2021-11-20 00:00:00,À NOITE,COMÉRCIO E SERVIÇOS,VILA MATIAS,ART. 213 - ESTUPRO,1,2022,NaN,NaN,NaN,NaN
1,SANTOS,2022-01-09,2022-01-09 00:00:00,DE MADRUGADA,RESIDÊNCIA,MARAPE,ESTUPRO DE VULNERAVEL (ART.217-A),1,2022,NaN,NaN,NaN,NaN
2,SANTOS,2022-01-16,2022-01-15 00:00:00,À NOITE,RESIDÊNCIA,MACUCO,ART. 213 - ESTUPRO,1,2022,NaN,NaN,NaN,NaN
3,SANTOS,2022-01-11,NaN,EM HORA INCERTA,RESIDÊNCIA,MORRO NOVA CINTRA,ESTUPRO DE VULNERAVEL (ART.217-A),1,2022,NaN,NaN,NaN,NaN
4,SANTOS,2022-02-05,2021-10-17 00:00:00,DE MADRUGADA,VIA PÚBLICA,MACUCO,ART. 213 - ESTUPRO,2,2022,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
221,SANTOS,2025-06-09,2025-03-15,A NOITE,RESIDÊNCIA,SABOO,ESTUPRO - ART. 213,6,2025,05º D.P. SANTOS,DEINTER 6 - SANTOS,DEL.SEC.SANTOS,SANTOS
222,SANTOS,2025-06-18,2024-08-30,EM HORA INCERTA,RESIDÊNCIA,CASTELO,ESTUPRO - ART. 213,6,2025,05º D.P. SANTOS,DEINTER 6 - SANTOS,DEL.SEC.SANTOS,SANTOS
223,SANTOS,2025-06-02,2025-06-02,A TARDE,RESIDÊNCIA,CHICO DE PAULA,ESTUPRO DE VULNERAVEL (ART.217-A),6,2025,05º D.P. SANTOS,DEINTER 6 - SANTOS,DEL.SEC.SANTOS,SANTOS
224,SANTOS,2025-06-17,2022-12-15,A NOITE,CONDOMÍNIO RESIDENCIAL,PIRATININGA,ESTUPRO DE VULNERAVEL (ART.217-A),6,2025,05º D.P. SANTOS,DEINTER 6 - SANTOS,DEL.SEC.SANTOS,SANTOS


# bairros

In [9]:
# normalizando os valores dos bairros

df_standard_time['BAIRRO'] = pd.DataFrame(unidecode(bairro.upper()) for bairro in df_standard_time['BAIRRO'])
df_standard_time['BAIRRO'].unique()

array(['VILA MATIAS', 'MARAPE', 'MACUCO', 'MORRO NOVA CINTRA', 'ESTUARIO',
       'APARECIDA', 'SANTA MARIA', 'GONZAGA', 'EMBARE', 'CHICO DE PAULA',
       'CASTELO', 'BOQUEIRAO', 'MORRO DE SAO BENTO', 'CENTRO', 'PAQUETA',
       'VILA SAO JORGE', 'AREA RURAL', 'PONTA DA PRAIA', 'CANELEIRA',
       'JOSE MENINO', 'ENCRUZILHADA', 'SAO MANOEL', 'VILA VOTURUA',
       'VILA NOVA', 'RADIO CLUBE', 'MORRO SANTA MARIA', 'BOM RETIRO',
       'CAMPO GRANDE', 'AREIA BRANCA', 'VILA PROGRESSO',
       'MORRO SAO BENTO', 'VILA MATHIAS', 'SABOO', 'PIRATININGA'],
      dtype=object)

In [10]:
# temos apenas dados de santos, então a coluna "NOMES_MUNICIPIO" não é mais necessária

df_new = df_standard_time.drop(columns='NOME_MUNICIPIO')
df_new

,DATA_REGISTRO,DATA_OCORRENCIA_BO,DESC_PERIODO,DESC_SUBTIPOLOCAL,BAIRRO,RUBRICA,MES_ESTATISTICA,ANO_ESTATISTICA,NOME_DELEGACIA_CIRCUNSCRICAO,NOME_DEPARTAMENTO_CIRCUNSCRICAO,NOME_SECCIONAL_CIRCUNSCRICAO,NOME_MUNICIPIO_CIRCUNSCRICAO
0,2022-01-05,2021-11-20 00:00:00,À NOITE,COMÉRCIO E SERVIÇOS,VILA MATIAS,ART. 213 - ESTUPRO,1,2022,NaN,NaN,NaN,NaN
1,2022-01-09,2022-01-09 00:00:00,DE MADRUGADA,RESIDÊNCIA,MARAPE,ESTUPRO DE VULNERAVEL (ART.217-A),1,2022,NaN,NaN,NaN,NaN
2,2022-01-16,2022-01-15 00:00:00,À NOITE,RESIDÊNCIA,MACUCO,ART. 213 - ESTUPRO,1,2022,NaN,NaN,NaN,NaN
3,2022-01-11,NaN,EM HORA INCERTA,RESIDÊNCIA,MORRO NOVA CINTRA,ESTUPRO DE VULNERAVEL (ART.217-A),1,2022,NaN,NaN,NaN,NaN
4,2022-02-05,2021-10-17 00:00:00,DE MADRUGADA,VIA PÚBLICA,MACUCO,ART. 213 - ESTUPRO,2,2022,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
221,2025-06-09,2025-03-15,A NOITE,RESIDÊNCIA,SABOO,ESTUPRO - ART. 213,6,2025,05º D.P. SANTOS,DEINTER 6 - SANTOS,DEL.SEC.SANTOS,SANTOS
222,2025-06-18,2024-08-30,EM HORA INCERTA,RESIDÊNCIA,CASTELO,ESTUPRO - ART. 213,6,2025,05º D.P. SANTOS,DEINTER 6 - SANTOS,DEL.SEC.SANTOS,SANTOS
223,2025-06-02,2025-06-02,A TARDE,RESIDÊNCIA,CHICO DE PAULA,ESTUPRO DE VULNERAVEL (ART.217-A),6,2025,05º D.P. SANTOS,DEINTER 6 - SANTOS,DEL.SEC.SANTOS,SANTOS
224,2025-06-17,2022-12-15,A NOITE,CONDOMÍNIO RESIDENCIAL,PIRATININGA,ESTUPRO DE VULNERAVEL (ART.217-A),6,2025,05º D.P. SANTOS,DEINTER 6 - SANTOS,DEL.SEC.SANTOS,SANTOS


# meses

In [11]:
# nomes dos meses num dicionario de conversão
# month names in a conversion dict

months = {
    1: 'JANEIRO',
    2: 'FEVEREIRO',
    3: 'MARÇO',
    4: 'ABRIL',
    5: 'MAIO',
    6: 'JUNHO',
    7: 'JULHO',
    8: 'AGOSTO',
    9: 'SETEMBRO',
    10: 'OUTUBRO',
    11: 'NOVEMBRO',
    12: 'DEZEMBRO',
}

df_new['MES'] = [months.get(n, "INCERTO") for n in df['MES_ESTATISTICA']]
df_new['MES']

,MES
0,JANEIRO
1,JANEIRO
2,JANEIRO
3,JANEIRO
4,FEVEREIRO
...,...
221,JUNHO
222,JUNHO
223,JUNHO
224,JUNHO


# datas

In [12]:
print(
    "occorence date: ", df_new['DATA_OCORRENCIA_BO'][0],
    "\n"
    "report date: ", df_new['DATA_REGISTRO'][0]
)

occorence date:  2021-11-20 00:00:00 
report date:  2022-01-05


PT-BR: para saber quanto tempo uma vitima levou para registrar uma ocorrência, podemos usar uma diferenciação
de data. porém, é preciso que ambas as datas estejam no mesmo formato. visualiza-se que o atributo da data de
ocorrência apresenta inconsistências de formato, então é preciso tratar isso.

EN: to find out how long it took a victm to file a report, we can use a date differentiation. however, both of
date must be in the same format. we see that the date of occurence feature has format inconsistences, so it needs
to be addressed.

In [13]:
# normalizando o formato das colunas data
# normalizing the date columns' format

def only_date(date_value: str, format: str = "%Y-%m-%d %H:%M:%S") -> str:
    """
    PT-BR: procedimento para extrair uma data de uma string num formato específico e a retornar no padrão ISO.
           ignora a data se já estiver no devido formato.

    EN: procediment to extract a date from a string in a especific format and return it like ISO standard.
        it ignores the date if it already is in the correct format.

    params
    ----------
    date_value : str

        PT-BR: data a ser formatada.
        EN: date to be formated.

    format : str

        PT-BR: formato específico da data de entrada. formato "%Y-%m-%d %H:%M:%S" como padrão.
        EN: input date's specified format. "%Y-%m-%d %H:%M:%S" as standard format.

    return:
    ----------
    str
        data formatada como %Y-%m-%d, padrão ISO.
        time formated like %Y-%m-%dm ISO standard.
    """
    
    if pd.isna(date_value):
        return None

    try:
        dt_format = datetime.strptime(date_value, format)
        dt = dt_format.date()

        return dt.isoformat()
    
    # considero que se o valor nao estiver no formato data + hora, então está no formato de apenas data
    # i am considering that if the value is not in date + hour format then it is in only date format
    except ValueError:
        return date_value


In [14]:
# data de ocorência normalizada
# occurrence date standardized

df_new['DATA_OCORRENCIA_BO'] = df_new['DATA_OCORRENCIA_BO'].apply(only_date)
df_new['DATA_OCORRENCIA_BO']

,DATA_OCORRENCIA_BO
0,2021-11-20
1,2022-01-09
2,2022-01-15
3,None
4,2021-10-17
...,...
221,2025-03-15
222,2024-08-30
223,2025-06-02
224,2022-12-15


In [15]:

week_day = []

for day in df_new['DATA_OCORRENCIA_BO']:
    if day == None:
        week_day.append(day)
        continue
    
    day = datetime.strptime(day, "%Y-%m-%d").weekday()
    week_day.append(day)

In [16]:
days_of_week = {
    0: "SEGUNDA", 
    1: "TERÇA",
    2: "QUARTA",
    3: "QUINTA",
    4: "SEXTA",
    5: "SABADO",
    6: "DOMINGO"
    }

df_new['DIA_SEMANA'] = [days_of_week.get(n, 'INCERTO') for n in week_day]
df_new['DIA_SEMANA']

,DIA_SEMANA
0,SABADO
1,DOMINGO
2,SABADO
3,INCERTO
4,DOMINGO
...,...
221,SABADO
222,SEXTA
223,SEGUNDA
224,QUINTA


In [17]:
# diferença entre as datas
# dates difference

def diff_days(d1, d2):
    """
    PT-BR: procedimento para calcular a diferença em dias entre duas datas no formato "%Y-%m-%d".
           caso alguma das datas seja nula, retorna 'INCERTO'.

    EN: procedure to calculate the difference in days between two dates in "%Y-%m-%d" format.
        if any of the dates is null, returns 'INCERTO'.

    params
    ----------
    d1 : str
        PT-BR: primeira data no formato "%Y-%m-%d".
        EN: first date in "%Y-%m-%d" format.

    d2 : str
        PT-BR: segunda data no formato "%Y-%m-%d".
        EN: second date in "%Y-%m-%d" format.

    return:
    ----------
    int | str
        PT-BR: diferença em dias entre as duas datas ou 'INCERTO' caso algum valor seja nulo.
        EN: difference in days between both dates or 'INCERTO' if any value is null.
    """

    for date in [d1, d2]:
        
        if pd.isna(date):
            return "INCERTO"

    # datetime objects
    d1 = datetime.strptime(d1, "%Y-%m-%d")
    d2 = datetime.strptime(d2, "%Y-%m-%d")

    return int((d2 - d1).days)


In [18]:
diff_days("2021-11-20", "2022-01-05")

46

In [19]:
# coluna para a diferença de dias
# days diferrence column

df_new['DIFERENCA_DIAS'] = df_new.apply(
    lambda row: diff_days(row['DATA_OCORRENCIA_BO'], row['DATA_REGISTRO']),
    axis=1
    )

df_new['DIFERENCA_DIAS']

,DIFERENCA_DIAS
0,46
1,0
2,1
3,INCERTO
4,111
...,...
221,86
222,292
223,0
224,915


In [20]:
# lista e tabela de frequência para o intevalo de dias
# list and frequency table to the interval of days

delta = []

for i in df_new['DIFERENCA_DIAS']:
    if i != "INCERTO":
        delta.append(i)

delta_table = pd.Series(delta).value_counts()
delta_table

,count
0,52
1,34
4,10
2,8
9,6
10,6
11,4
8,4
3,4
6,4


In [21]:
# para facilitar a contagem de ocorrêcnias e os gráficos
# to facilite the count of occurences and graphs

df_new['N'] = [1 for n in range(0, len(df_new))]
df_new['N']

,N
0,1
1,1
2,1
3,1
4,1
...,...
221,1
222,1
223,1
224,1


# save

In [22]:
'''
as culunas de datas foram úteis no processo de limpeza, de tratamento e de extração de novas
features, mas não serão mais relevantes para as visualizações, então elas serão dropadas antes
do carregamento.
'''

df_final = df_new.drop(columns=['DATA_REGISTRO', 'DATA_OCORRENCIA_BO'])

In [23]:
# dataframe
df_final.tail()

,DESC_PERIODO,DESC_SUBTIPOLOCAL,BAIRRO,RUBRICA,MES_ESTATISTICA,ANO_ESTATISTICA,NOME_DELEGACIA_CIRCUNSCRICAO,NOME_DEPARTAMENTO_CIRCUNSCRICAO,NOME_SECCIONAL_CIRCUNSCRICAO,NOME_MUNICIPIO_CIRCUNSCRICAO,MES,DIA_SEMANA,DIFERENCA_DIAS,N
221,A NOITE,RESIDÊNCIA,SABOO,ESTUPRO - ART. 213,6,2025,05º D.P. SANTOS,DEINTER 6 - SANTOS,DEL.SEC.SANTOS,SANTOS,JUNHO,SABADO,86,1
222,EM HORA INCERTA,RESIDÊNCIA,CASTELO,ESTUPRO - ART. 213,6,2025,05º D.P. SANTOS,DEINTER 6 - SANTOS,DEL.SEC.SANTOS,SANTOS,JUNHO,SEXTA,292,1
223,A TARDE,RESIDÊNCIA,CHICO DE PAULA,ESTUPRO DE VULNERAVEL (ART.217-A),6,2025,05º D.P. SANTOS,DEINTER 6 - SANTOS,DEL.SEC.SANTOS,SANTOS,JUNHO,SEGUNDA,0,1
224,A NOITE,CONDOMÍNIO RESIDENCIAL,PIRATININGA,ESTUPRO DE VULNERAVEL (ART.217-A),6,2025,05º D.P. SANTOS,DEINTER 6 - SANTOS,DEL.SEC.SANTOS,SANTOS,JUNHO,QUINTA,915,1
225,DE MADRUGADA,VIA PÚBLICA,JOSE MENINO,ESTUPRO - ART. 213,6,2025,07º D.P. SANTOS,DEINTER 6 - SANTOS,DEL.SEC.SANTOS,SANTOS,JUNHO,QUARTA,0,1


In [24]:
df_final.to_csv("/content/drive/My Drive/projeto-integrador-I/data/processed/df.csv", index=False)
delta_table.to_csv("/content/drive/My Drive/projeto-integrador-I/data/processed/delta_table.csv")